# Notebook with the code necessary to perform the baseline approach experimentation of NAS to solve AV

Luis Ignacio Ferro Salinas

## Hardware info

Simple commands to get the information about the hardware on colab

In [ ]:
#!cat /etc/*release

In [ ]:
#!df -h

In [ ]:
#!cat /proc/cpuinfo

In [ ]:
#!cat /proc/meminfo

## Notebook setup

Dependencies

In [ ]:
%%capture
!pip install -q gwpy
!pip install transformers[torch]
!pip install datasets
!pip install nltk
!pip install kneed
!pip install accelerate==0.24.1


In [ ]:
def debug(thing, title):
    print("-------------------------------------------------------------------")
    print(title)
    print(thing)
    print("-------------------------------------------------------------------")

In [ ]:
from zipfile import ZipFile
from collections import Counter
import os
import json
import random
import pandas as pd
import numpy as np
import math
from datasets import Dataset, Features, Value, ClassLabel, IterableDataset
from transformers import AutoTokenizer
import torch
from transformers import AutoModel
from transformers import Trainer
import numpy as np
import itertools
from sklearn.linear_model import LogisticRegression
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
import warnings
warnings.filterwarnings(action = 'ignore')
import gensim
from gensim.models import Word2Vec
from kneed import KneeLocator
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB, \
    BernoulliNB, CategoricalNB
from sklearn.neural_network import MLPClassifier
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree
import pickle
import string
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import classification_report
import time
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.fixes import loguniform
from tqdm.auto import trange
from sklearn.metrics import roc_curve, auc
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

In [ ]:
%%capture
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
path_to_save = os.path.join('drive', 'MyDrive', 'thesis', 'Preprocessed')

## Helper functions

This section consolidates a suite of modular functions designed to streamline and automate various stages of text processing, data preparation, and feature extraction, specifically tailored for machine learning experimentation with text data. These functions handle tasks ranging from data splitting and text preprocessing to embedding extraction using BERT, promoting code reusability and simplifying complex workflows.

Splits the input documents and labels into training and testing sets (X_train, X_test, y_train, y_test), facilitating the evaluation of model performance on unseen data.

In [ ]:
def split_data(training_data, testing_data):
    X_train = []
    y_train = []
    X_test = []
    y_test = []

    for i in range(len(training_data.index)):
        row = training_data.iloc[i]
        X_train += [row['text1']]
        X_train += [row['text2']]
        y_train += [row['label']]

    for i in range(len(testing_data.index)):
        row = testing_data.iloc[i]
        X_test += [row['text1']]
        X_test += [row['text2']]
        y_test += [row['label']]
    return X_train, y_train, X_test, y_test

Preprocesses a list of documents by converting all text to lowercase and then segmenting it into individual tokens (words or subwords), a standard step in natural language processing.

In [ ]:
def word_tokenize_docs(X, y):
    data = []

    for document in X:
        document = document.replace("\n", " ")
        temp = []
        for cap_tword in word_tokenize(document):
            temp.append(cap_tword.lower())

        data.append(temp)
    return data, y

Divides a given list of words into smaller sequences (chunks) of a specified length, enabling the processing of long texts in manageable segments.

In [ ]:
def split_chunks(word_list, chunk_length):
    chunk_ints = np.arange(0, len(word_list), chunk_length)
    #debug(chunk_ints, 'ints')

    chunks = []

    for i, start_int in enumerate(chunk_ints):
        if i == len(chunk_ints) - 1:
            chunki = ' '.join(word_list[start_int:])
            chunks += [chunki]
            #debug(len(chunki), 'lenchunk')
            continue
        end_int = chunk_ints[i + 1]
        chunki = ' '.join(word_list[start_int:end_int])
        chunks += [chunki]
        #debug(len(chunki), 'len chunk')

    return chunks, len(chunks)

Applies the text chunking function to an entire corpus, generating a list of chunked documents, preparing the data for chunk-based analysis.

In [ ]:
def split_dataset(x, y, chunk_length):

    X = []

    chunks_pdoc = []

    for i in range(0, len(x) - 1, 2):
        text1_chunks, text1_nchunks = split_chunks(x[i], chunk_length)
        text2_chunks, text2_nchunks = split_chunks(x[i + 1], chunk_length)

        for t1_chunk in text1_chunks:
            X += [t1_chunk]

        chunks_pdoc += [text1_nchunks]

        for t2_chunk in text2_chunks:
            X += [t2_chunk]

        chunks_pdoc += [text2_nchunks]

    """Y = []
    for author1, author2 in y:
        for i in range(4):
            Y += [author1]
        for i in range(4):
            Y += [author2]"""
    # Chunks per document.
    return X, y, chunks_pdoc

Computes a similarity score by generating embeddings for text chunks, aggregating these embeddings, and then comparing the resulting vector between embeddings of known and unknown documents.

In [ ]:
# compare embeddings of texts chunked
def compare_eots_ch(X_e, chunks_per_doc, sub_order):
    X_doc_embs = []
    X_compared = []

    chunk_shape = X_e[0].shape
    debug(chunk_shape, 'chunk shape')

    i = 0
    # Current chunks per document
    for curr_cpd in chunks_per_doc:
        p1 = np.zeros(chunk_shape)

        for j in range(curr_cpd):
            p1 = np.add(p1, X_e[i + j])

        X_doc_embs += [p1]

        # cool
        i = i + curr_cpd

    for k in range(0, len(X_doc_embs) - 1, 2):
        known_emb = X_doc_embs[k]
        unknown_emb = X_doc_embs[k + 1]

        if sub_order == 1:
            l = np.subtract(known_emb, unknown_emb)
        elif sub_order == 2:
            l = np.subtract(unknown_emb, known_emb)
        X_compared += [l]
    return np.array(X_compared)


Organizes the input text data and corresponding labels into a Pandas DataFrame, providing a structured format for data manipulation and analysis.

In [ ]:
def make_df(X, Y):
    X = pd.DataFrame(X, columns=['text'])
    Y = pd.DataFrame(Y, columns=['label'])

    df = pd.DataFrame()
    df['text'] = X
    df['label'] = Y

    df = df.fillna(-1)
    df['label'] = df['label'].astype('int')
    return df

Transforms the DataFrame into an iterable dataset, optimized for efficient processing with BERT models, particularly beneficial for large datasets of for resource-constrained environments.

In [ ]:
def my_generator(n, df):
        for i in range(len(df.index)):
            yield {"text": df['text'].iloc[i], "label": df['label'].iloc[i]}

def create_iterableds(df):
    #class_names = list(df['label'].unique())
    data_features = Features({'text': Value('string'),
                              'label': ClassLabel(names=[1, 0, -1])})
    n_examples = df.shape[0]
    my_iterable_dataset = IterableDataset.from_generator(my_generator,
                                gen_kwargs={"n": n_examples, "df": df},
                                features=data_features,)
    return my_iterable_dataset

Converts the DataFrame into a standard PyTorch Dataset, suitable for general machine learning tasks, but loads entirely into memory.

In [ ]:
def create_torchds(df):
    class_names = list(df['label'].unique())

    data_features = Features({'text': Value('string'), 'label': ClassLabel(names=class_names)})

    train_dataset = Dataset.from_pandas(df, split='train', features=data_features)
    print(train_dataset.features)
    return train_dataset

Applies a BERT tokenizer to a batch of text data from the DataFrame, converting the text into numerical tokens that the BERT model can understand.

In [ ]:
def tokenize(batch):
    return tokenizer(batch["text"], max_length=512, padding='max_length', truncation=True)

Converts a batch of tokenized data into tensors, the fundamental data structure used for computation in PyTorch, enabling efficient processing by the BERT model.

In [ ]:
# make each example a torch tensors
def tensorize(batch):
    #debug(len(batch), 'batch')
    tensorized_batch = {}
    for key in batch:
        if key != 'text':
            tensorized_batch[key] = np.array(batch[key])
        else:
            tensorized_batch[key] = batch[key]
    return tensorized_batch

Processes a batch of data through a pre-trained BERT model and extracts the final layer's hidden states, which serve as rich contextualized representations of the input text.

In [ ]:
def extract_hidden_states(batch):
    # Place model inputs on the GPU
    print(batch.keys())
    inputs = {k:torch.as_tensor(v).to(device) for k,v in batch.items()
        if k in tokenizer.model_input_names} # Extract last hidden states
    with torch.no_grad():
        last_hidden_state = model(**inputs).last_hidden_state
        # Return vector for [CLS] token

    #print(last_hidden_state)
    return {"hidden_state": last_hidden_state[:,0].cpu().numpy()}

 Integrates several preceding functions to perform end-to-end processing of the entire DataFrame: tokenization, tensorization, dataset formatting, BERT-based feature extraction (hidden states), and conversion to a NumPy array for streamlined analysis.

In [ ]:
# Extract features from texts with pretrained BERT using iterable dataset
def extract_features(df):
    ids = create_iterableds(df)
    texts_tokenized = ids.map(tokenize, batched=True, batch_size=2, input_columns=None)
    texts_tensorized = texts_tokenized.map(tensorize, batched=True,
                                           batch_size=2)
    texts_informat = texts_tensorized.remove_columns(['text',])
    last_hiddens_train = texts_informat.map(extract_hidden_states, batched=True,
                                            batch_size=1, input_columns=None)
    #last_hiddens_train = texts_informat.map(extract_hidden_states, batched=True,
    #                                        batch_size=1)
    last_hiddens_list = []

    for ind, last_hidden in enumerate(last_hiddens_train):
        debug('', f'extracting features frome example: {ind}')
        last_hiddens_list += [last_hidden['hidden_state']]
    X_e = np.array(last_hiddens_list) # X embeddings.
    return X_e

This function is central to the authorship verification process. It operates on a DataFrame of chunked documents, using the provided chunking structure to extract feature representations for corresponding chunks within known and unknown documents.  It then performs a comparative analysis (e.g., by subtracting) of these chunk features, adhering to a specified order (known vs. unknown), to generate a set of vectors. Each output vector encapsulates the stylistic differences or similarities between the known and unknown documents, providing crucial information for determining common authorship.

In [ ]:

def compare_docs_embeddings(df, cpd, sub_order):
    # Chunks per document

    X_c = []
    # Previous chunks per document sum
    prev_cpds = 0
    # Current chunks per document
    n_docs = len(cpd)

    for j in range(0, len(cpd) - 1, 2):

        debug(j, f'Current starting ind out of {n_docs}')

        ckcpd = cpd[j]
        cucpd = cpd[j + 1]

        debug(ckcpd, 'current chunks per known doc')
        debug(cucpd, 'current chunks per unknown doc')

        #current X embedding
        cX_e = extract_features(df[prev_cpds:prev_cpds + ckcpd + cucpd])

        # Current X compared
        cX_c = compare_eots_ch(cX_e, [ckcpd, cucpd], sub_order)

        X_c += [cX_c]

        prev_cpds += ckcpd + cucpd

        #X_testc = compare_eots_ch(X_te, test_cpd)
    return X_c


Load BERT components

In [ ]:
model_ckpt = 'distilbert-base-uncased'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_ckpt).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Function to load the data and return the training/test features of a complete dataset.

In [ ]:
def bigBoy(year, language, chunk_size, sub_order):
    # sub order 1: known - unknown, 2: unknown - known

    start_time = time.time()
    if year == 2020:
        # s for small it's not the entire dataset too large
        training_datas = pd.read_csv(os.path.join(path_to_save,
                                             'training_data2020s.csv'), sep='\\')
        testing_datas = pd.read_csv(os.path.join(path_to_save,
                                            'testing_data2020s.csv'), sep='\\')
    elif year == 2015:
        if language == "En":
            training_data = pd.read_csv(os.path.join(path_to_save,
                                                'training_data2015En.csv'), sep='\\')
            testing_data = pd.read_csv(os.path.join(path_to_save,
                                                'testing_data2015En.csv'), sep='\\')

        elif language == "Es":
            training_data = pd.read_csv(os.path.join(path_to_save,
                                                'training_data2015Es.csv'), sep='\\')
            testing_data = pd.read_csv(os.path.join(path_to_save,
                                                'testing_data2015Es.csv'), sep='\\')

    if year == 2020:
        X_train, y_train, X_test, y_test = \
            split_data(training_datas, testing_datas)
    elif year == 2015:
        X_train, y_train, X_test, y_test = \
            split_data(training_data, testing_data)

    X_train, y_train = word_tokenize_docs(X_train, y_train)
    X_test, y_test = word_tokenize_docs(X_test, y_test)

    # cpd Chunks per document
    X_trainch, y_train, training_cpd= split_dataset(X_train, y_train, chunk_size)
    X_testch, y_test, test_cpd = split_dataset(X_test, y_test, chunk_size)

    X_train, y_train = X_trainch, y_train
    X_test, y_test = X_testch, y_test

    train_df = make_df(X_train, y_train)

    X_trainc = compare_docs_embeddings(train_df, training_cpd, sub_order)

    test_df = make_df(X_test, y_test)

    X_te = compare_docs_embeddings(test_df, test_cpd, sub_order)

    return np.array(X_trainc).reshape(len(X_trainc), 768), \
           np.array(X_te).reshape(len(X_te), 768), \
           time.time() - start_time


## Generate embeddings files

This section focuses on the systematic extraction and storage of document features across the entire collection of datasets.  It leverages the previously defined function to process each dataset, generating a set of feature vectors representing the stylistic characteristics of the documents. The resulting embeddings for all datasets are then efficiently saved as NumPy (.npy) files. This process ensures that the extracted features are readily available and optimized for subsequent modeling and analysis, facilitating efficient experimentation across multiple datasets.

In [ ]:
year = 2020
language = 'En'
chunk_size = 500
sub_order = 1


X_trainc, X_te, elapsed_time = bigBoy(year, language, chunk_size, sub_order)
sub_ordert = 'A-B' if sub_order == 1 else 'B-A'

# Path to save embeddings
path_tse = os.path.join('drive', 'MyDrive', 'thesis', 'ProblemEmbeddings',
                        str(year), language, sub_ordert)

# Path to save embeddings training
path_tset = os.path.join(path_tse, f'{year}{language}{sub_ordert}{chunk_size}')

np.save(path_tset + 'Training', X_trainc)
np.save(path_tset + 'Test', X_te)

with open (path_tset + 'time.txt', 'w') as time_file:
    time_file.write(f'Execution time: {elapsed_time} seconds')

In [ ]:
debug(X_trainc.shape, 'X train compared shape')
debug(X_te.shape, 'X test compared shape')

-------------------------------------------------------------------
X train compared shape
(70, 768)
-------------------------------------------------------------------
-------------------------------------------------------------------
X test compared shape
(30, 768)
-------------------------------------------------------------------


In [ ]:
X_trainc[0][0:5]

array([-1.49714342,  2.33236619,  0.59767171, -0.62748791,  0.99508618])

In [ ]:
X_train_reloaded = np.load(path_tset + 'Training.npy')

In [ ]:
X_train_reloaded[0][0:5]

array([-1.49714342,  2.33236619,  0.59767171, -0.62748791,  0.99508618])

In [ ]:
X_trainc.shape

(70, 768)

In [ ]:
X_train_reloaded.shape

(70, 768)

In [ ]:
X_te_reloaded = np.load(path_tset + 'Test.npy')

In [ ]:
X_te[0][0:5]

array([-0.50045844, -0.06271456, -0.15127962,  1.17925267, -2.24663673])

In [ ]:
X_te_reloaded[0][0:5]

array([-0.50045844, -0.06271456, -0.15127962,  1.17925267, -2.24663673])

In [ ]:
debug(X_te.shape, 'Shape of test embeddings')
debug(X_te_reloaded.shape, 'Shape of test reloaded embeddings')

-------------------------------------------------------------------
Shape of test embeddings
(30, 768)
-------------------------------------------------------------------
-------------------------------------------------------------------
Shape of test reloaded embeddings
(30, 768)
-------------------------------------------------------------------


## Traditional classifiers vs. baseline NAS

Calculate the AUC using a confusion matrix

In [ ]:
def g(x, y):
    if y > 0:
        return x / y
    else:
        return 1

def calc_auc(conf_mat):
    return (g(conf_mat[0, 0], conf_mat[0, 0] + conf_mat[0, 1]) + g(conf_mat[1, 1], conf_mat[1, 1] + conf_mat[1, 0])) / 2

In [ ]:
conf_mat_ex = np.array([np.array([1, 2]), np.array([3, 4])])
conf_mat_ex

array([[1, 2],
       [3, 4]])

In [ ]:
calc_auc(conf_mat_ex)

0.45238095238095233

This function retrieves pre-computed document embeddings from stored NumPy (.npy) files. It takes dataset-specific information, including year, language, chunk size, and subtraction order, as input to dynamically locate and load the relevant embedding file from the cloud. This modular approach allows for flexible access to pre-processed features, ensuring that the correct data representations are used for downstream analysis or modeling tasks.

In [ ]:
def load_X_Y(year, language, chunk_size, sub_order):
    # At this point we only use A-B we get the other B-A in classifiers
    sub_ordert = 'A-B'

    # Path to save embeddings
    path_tse = os.path.join('drive', 'MyDrive', 'thesis', 'ProblemEmbeddings',
                            str(year), language, sub_ordert)

    # Path to save embeddings training
    path_tset = os.path.join(path_tse, f'{year}{language}{sub_ordert}{chunk_size}')

    X_trainc_reloaded = np.load(path_tset + 'Training.npy')
    X_testc_reloaded = np.load(path_tset + 'Test.npy')

    if year == 2020:
        # s for small it's not the entire dataset too large
        training_datas = pd.read_csv(os.path.join(path_to_save,
                                             'training_data2020s.csv'), sep='\\')
        testing_datas = pd.read_csv(os.path.join(path_to_save,
                                            'testing_data2020s.csv'), sep='\\')
    elif year == 2015:
        if language == "En":
            training_data = pd.read_csv(os.path.join(path_to_save,
                                                'training_data2015En.csv'), sep='\\')
            testing_data = pd.read_csv(os.path.join(path_to_save,
                                                'testing_data2015En.csv'), sep='\\')

        elif language == "Es":
            training_data = pd.read_csv(os.path.join(path_to_save,
                                                'training_data2015Es.csv'), sep='\\')
            testing_data = pd.read_csv(os.path.join(path_to_save,
                                                'testing_data2015Es.csv'), sep='\\')

    if year == 2020:
        X_train, y_train, X_test, y_test = \
            split_data(training_datas, testing_datas)
    elif year == 2015:
        X_train, y_train, X_test, y_test = \
            split_data(training_data, testing_data)

    return X_trainc_reloaded, y_train, X_testc_reloaded, y_test



Baseline NAS

This function automates a simplified Neural Architecture Search (NAS) procedure. Given a dataset's features, it systematically explores a predefined hyperparameter space, including the number of hidden layers, the number of neurons per layer, activation functions, and solvers. For each unique combination, it trains and evaluates a model, recording the resulting performance metrics (e.g., accuracy, F1-score). Finally, it compiles these measurements into a DataFrame, providing a comprehensive overview of the hyperparameter search and enabling informed selection of optimal configurations.

In [ ]:
def baby_nas(X_trainc, y_train, X_testc, y_test, sub_order):

    start_time = time.time()

    if sub_order == 2:
        X_trainc, X_testc = -X_trainc, -X_testc

    selected_arch = None
    max_acc = -1
    best_preds = []
    best_act = ''
    best_solver = ''
    best_model = None
    n_neurons = [4, 8, 16, 32, 64, 128]

    experiments_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy'])

    for classifier_head in list(itertools.combinations_with_replacement(n_neurons, 1)) + \
            list(itertools.combinations_with_replacement(n_neurons, 2)) + \
            list(itertools.combinations_with_replacement(n_neurons, 3)):
        classifier_head = list(classifier_head) + [1]
        for activation_name in ['relu', 'logistic', 'tanh', 'identity']:
            for solver_name in ['lbfgs','sgd', 'adam']:
                clf = MLPClassifier(hidden_layer_sizes=classifier_head, activation=activation_name, solver=solver_name)
                #clf = DecisionTreeClassifier(random_state=0)
                clf.fit(X_trainc, y_train)
                #cross_val_score(clf, X_t, Y_t, cv=10)
                preds = clf.predict(X_testc)

                report_dict = classification_report(y_test, preds, output_dict=True)
                try:
                    report_dict['1'] = report_dict['1.0']
                except:
                    ...
                conf_mat = confusion_matrix(y_test, preds)
                auc = calc_auc(conf_mat)

                #debug(report_dict, 'report dictionary obtained')

                curr_exp_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy', 'auc', 'features', 'training time (s)'])
                curr_exp_df['precision'] = [report_dict['1']['precision']]
                curr_exp_df['recall'] = [report_dict['1']['recall']]
                curr_exp_df['f1-score'] = [report_dict['1']['f1-score']]
                curr_exp_df['accuracy'] = [report_dict['accuracy']]
                curr_exp_df['auc'] = [auc]
                curr_exp_df['features'] = [f'MLP with solver {solver_name}, \
                    activation {activation_name} in hidden layers and \
                    architecture {classifier_head}']

                experiments_df = pd.concat([experiments_df, curr_exp_df])

                #acc = accuracy_score(y_test, preds)
                """if acc > max_acc:
                    max_acc = acc
                    selected_arch = classifier_head
                    best_preds = preds
                    best_act = activation_name
                    best_solver = solver_name
                    best_model = clf"""
    experiments_df['training time (s)'] = time.time() - start_time

    return experiments_df.sort_values('auc', ascending=False)





 Logistic Regression

Trains a Logistic Regression model with default hyperparameters on the dataset's features and saves the resulting performance metrics in a DataFrame.

In [ ]:
def log_reg_simple(X_trainc, y_train, X_testc, y_test, sub_order):
    start_time = time.time()

    if sub_order == 2:
        X_trainc, X_testc = -X_trainc, -X_testc

    my_log_reg = LogisticRegression()
    my_log_reg.fit(X_trainc, y_train)

    preds = my_log_reg.predict(X_testc)

    report_dict = classification_report(y_test, preds, output_dict=True)
    try:
        report_dict['1'] = report_dict['1.0']
    except:
        ...
    conf_mat = confusion_matrix(y_test, preds)
    debug(conf_mat, 'confusion mat')
    auc = calc_auc(conf_mat)

    #debug(report_dict, 'report dict')

    experiments_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy', 'auc', 'features', 'training time (s)'])
    experiments_df['precision'] = [report_dict['1']['precision']]
    experiments_df['recall'] = [report_dict['1']['recall']]
    experiments_df['f1-score'] = [report_dict['1']['f1-score']]
    experiments_df['accuracy'] = [report_dict['accuracy']]
    experiments_df['auc'] = [auc]
    experiments_df['features'] = ['Logistic Regression Classifier with default parameters']
    experiments_df['training time (s)'] = [time.time() - start_time]

    return experiments_df

Support Vector Machine

 Trains a Support Vector Machine (SVM) model with default hyperparameters on the dataset's features and saves the resulting performance metrics in a DataFrame.

In [ ]:
def svm(X_trainc, y_train, X_testc, y_test, sub_order):
    start_time = time.time()

    if sub_order == 2:
        X_trainc, X_testc = -X_trainc, -X_testc

    my_svm = SVC()
    my_svm.fit(X_trainc, y_train)

    preds = my_svm.predict(X_testc)

    report_dict = classification_report(y_test, preds, output_dict=True)
    try:
        report_dict['1'] = report_dict['1.0']
    except:
        ...
    conf_mat = confusion_matrix(y_test, preds)
    auc = calc_auc(conf_mat)

    experiments_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy', 'auc', 'features', 'training time (s)'])
    experiments_df['precision'] = [report_dict['1']['precision']]
    experiments_df['recall'] = [report_dict['1']['recall']]
    experiments_df['f1-score'] = [report_dict['1']['f1-score']]
    experiments_df['accuracy'] = [report_dict['accuracy']]
    experiments_df['auc'] = [auc]
    experiments_df['features'] = ['Default SVC SVM']
    experiments_df['training time (s)'] = [time.time() - start_time]

    return experiments_df

naive Bayes

Trains a Naive Bayes model with default hyperparameters on the dataset's features and saves the resulting performance metrics in a DataFrame.

In [ ]:
def naive_bayes(X_trainc, y_train, X_testc, y_test, sub_order):
    start_time = time.time()

    if sub_order == 2:
        X_trainc, X_testc = -X_trainc, -X_testc

    my_nb = BernoulliNB()
    my_nb.fit(X_trainc, y_train)

    preds = my_nb.predict(X_testc)

    report_dict = classification_report(y_test, preds, output_dict=True)
    try:
        report_dict['1'] = report_dict['1.0']
    except:
        ...
    conf_mat = confusion_matrix(y_test, preds)
    auc = calc_auc(conf_mat)

    experiments_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy', 'auc', 'features', 'training time (s)'])
    experiments_df['precision'] = [report_dict['1']['precision']]
    experiments_df['recall'] = [report_dict['1']['recall']]
    experiments_df['f1-score'] = [report_dict['1']['f1-score']]
    experiments_df['accuracy'] = [report_dict['accuracy']]
    experiments_df['auc'] = [auc]
    experiments_df['features'] = ['Default Bernoulli naive Bayes']
    experiments_df['training time (s)'] = [time.time() - start_time]

    return experiments_df

Example training of a Logistic regression

In [ ]:

X_trainc, y_train, X_testc, y_test = load_X_Y(2015, 'En', chunk_size='200', sub_order=1)
log_reg_ex = log_reg_simple(X_trainc, y_train, X_testc, y_test, sub_order=1)
log_reg_ex


-------------------------------------------------------------------
confusion mat
[[112 138]
 [154  96]]
-------------------------------------------------------------------


,precision,recall,f1-score,accuracy,auc,features,training time (s)
0,0.410256,0.384,0.396694,0.416,0.416,Logistic Regression Classifier with default pa...,0.147045


## Execution of experiments

This function serves as the central orchestrator for the experiment. Given a dataset description, it performs the following sequence of actions: 1) loads the corresponding pre-computed features; 2) executes a simplified Neural Architecture Search (baseline NAS) to explore potential model configurations; 3) trains and evaluates Logistic Regression, SVM, and Naive Bayes classifiers using their default settings; 4) consolidates the performance metrics from all experiments (NAS and the three classifiers) into a single DataFrame; and 5) saves this DataFrame to a CSV file in cloud storage, ensuring persistent and accessible results.

In [ ]:
def run_experiments(year, language, chunk_size, sub_order):
    # sub order 1 -> known - unknown
    # sub order 2 -> unknown - known
    sub_ordert = 'A-B' if sub_order == 1 else 'B-A'

    X_trainc, y_train, X_testc, y_test = load_X_Y(year, language, chunk_size, sub_order)


    # Nas experiments df
    nas_edf = baby_nas(X_trainc, y_train, X_testc, y_test, sub_order)
    log_reg_edf = log_reg_simple(X_trainc, y_train, X_testc, y_test, sub_order)
    svm_edf = svm(X_trainc, y_train, X_testc, y_test, sub_order)
    nb_edf = naive_bayes(X_trainc, y_train, X_testc, y_test, sub_order)

    experiments_df = pd.concat([nas_edf, log_reg_edf, svm_edf, nb_edf])

    experiments_df.sort_values('auc', ascending=False, inplace=True)

    # Path to save results
    path_tsr = os.path.join('drive', 'MyDrive', 'thesis', 'MetricResults',
                                str(year), language, sub_ordert)
    experiments_df.to_csv(os.path.join(path_tsr, f'{str(year)}{language}{sub_ordert}{chunk_size}Results.csv'), index=False)

    return


We simply use the previous function for all defined datasets.

In [ ]:
possible_languages = ['En', 'Es']
possible_years = [2015, 2020]
possible_chunk_size = [100, 200, 300, 400, 500]
possible_sub_order = [1, 2]

a = itertools.product(possible_chunk_size, [2020], ['En'], possible_sub_order)
b = itertools.product([100, 200], [2015], ['En'], possible_sub_order)
c = itertools.product(possible_chunk_size, [2015], ['Es'], possible_sub_order)

experiment_variations = list(a) + list(b) + list(c)

#debug(experiment_variations, f'{len(experiment_variations)}')

for experiment_variation in experiment_variations:
    chunk_size, year, language, sub_order = experiment_variation

    debug(f'Year {year} language {language} chunk size {chunk_size} subtraction order {sub_order}', 'Experiment variation')
    run_experiments(year, language, chunk_size, sub_order)


-------------------------------------------------------------------
Experiment variation
Year 2020 language En chunk size 100 subtraction order 1
-------------------------------------------------------------------
-------------------------------------------------------------------
Experiment variation
Year 2020 language En chunk size 100 subtraction order 2
-------------------------------------------------------------------
-------------------------------------------------------------------
Experiment variation
Year 2020 language En chunk size 200 subtraction order 1
-------------------------------------------------------------------
-------------------------------------------------------------------
Experiment variation
Year 2020 language En chunk size 200 subtraction order 2
-------------------------------------------------------------------
-------------------------------------------------------------------
Experiment variation
Year 2020 language En chunk size 300 subtraction order 

In [ ]:

debug(classification_report(preds_nas, y_test), f'MLP NAS execution time: {time_nas} seconds')
debug(classification_report(preds_log_reg, y_test), f'Logistic regression execution time: {time_log_reg} seconds')
debug(classification_report(preds_svm, y_test), f'SVM execution time: {time_svm} seconds')
debug(classification_report(preds_nb, y_test), f'naive Bayes execution time: {time_nb} seconds')

## Add regular logistic regression

Originally, we used a more complex version of logistic regression defined in the literature, but after completing the experiments, we decided that the LR with default parameters would be more consistent with the other traditional approaches, so the results from this default LR were added afterwards.

In [ ]:
possible_languages = ['En', 'Es']
possible_years = [2015, 2020]
possible_chunk_size = [100, 200, 300, 400, 500]
possible_sub_order = [1, 2]

a = itertools.product(possible_chunk_size, [2020], ['En'], possible_sub_order)
b = itertools.product([100, 200], [2015], ['En'], possible_sub_order)
c = itertools.product(possible_chunk_size, [2015], ['Es'], possible_sub_order)

experiment_variations =  list(b) + list(c)

for chunk_size, year, language, sub_order in experiment_variations:
    sub_ordert = 'A-B' if sub_order == 1 else 'B-A'

    X_trainc, y_train, X_testc, y_test = load_X_Y(year, language, chunk_size, sub_order)

    # Simple logistic regression df.
    simple_lrdf = log_reg_simple(X_trainc, y_train, X_testc, y_test, sub_order)

    # Path to save results
    path_tsr = os.path.join('drive', 'MyDrive', 'thesis', 'MetricResults',
                            str(year), language, sub_ordert)
    simple_lrdf.to_csv(os.path.join(path_tsr, f'{str(year)}{language}{sub_ordert}{chunk_size}SimpleLRResults.csv'), index=False)






## Get best results per data collection

 This following part consolidates the results of multiple experiments conducted across various datasets. For each dataset, it processes the individual CSV files generated by different experimental configurations. It enriches the data by adding a new column that specifies the exact configuration of each experiment (e.g., hyperparameters, model type). Finally, it concatenates all these experiment variations into a single, comprehensive DataFrame, providing a unified view of all experimental outcomes for subsequent analysis and comparison.

In [ ]:
# Add sub_order to features
def add_subotf(features_as_row, sub_order):
    features_str = features_as_row.values[0]
    return f'{sub_order} {features_str}'

def add_chunks(features_as_row, chunk_size):
    features_str = features_as_row.values[0]
    return f'Chunks {str(chunk_size)} {features_str}'

# get top results per collection

def get_trpc():
    # Dfs per collection
    dfs_pcollection = []
    possible_languages = ['En', 'Es']
    possible_years = [2015, 2020]

    # Possible chunks per data collection
    possible_chunkspdc = {(2020, 'En'): [100, 200, 300, 400, 500], \
        (2015, 'En'): [100, 200], (2015, 'Es'): [100, 200, 300, 400, 500]}

    data_collections = [(2020, 'En'), (2015, 'En'), (2015, 'Es')]

    for year, language in data_collections:

        data_collectiondf = pd.DataFrame()

        for chunk_size in possible_chunkspdc[(year, language)]:
            for sub_order in [1, 2]:
                sub_ordert = 'A-B' if sub_order == 1 else 'B-A'

                # Path to save results
                path_tsr = os.path.join('drive', 'MyDrive', 'thesis', 'MetricResults',
                                    str(year), language, sub_ordert)

                #simple_lrdf = pd.read_csv(os.path.join(path_tsr, f'{str(year)}{language}{sub_ordert}{chunk_size}SimpleLRResults.csv'))
                #simple_lrdf['features'] = simple_lrdf[['features']].apply(add_subotf, axis=1, args=([sub_ordert]))
                #simple_lrdf['features'] = simple_lrdf[['features']].apply(add_chunks, axis=1, args=([chunk_size]))

                experiments_df = pd.read_csv(os.path.join(path_tsr, f'{str(year)}{language}{sub_ordert}{chunk_size}Results.csv'))
                experiments_df['features'] = experiments_df[['features']].apply(add_subotf, axis=1, args=([sub_ordert]))
                experiments_df['features'] = experiments_df[['features']].apply(add_chunks, axis=1, args=([chunk_size]))

                #debug(experiments_df, 'experiments df without simple log reg')

                #experiments_df = pd.concat([experiments_df, simple_lrdf])

                #debug(experiments_df, 'experiments df for this chunk size')

                data_collectiondf = pd.concat([data_collectiondf, experiments_df])

        #debug(data_collectiondf, 'For given data collection, this is the experiments')
        dfs_pcollection += [data_collectiondf.sort_values('auc', ascending=False)]
    return dfs_pcollection



In [ ]:
final_r2020En, final_r2015En, final_r2015Es = get_trpc()

In [ ]:
debug(final_r2020En.info(), 'info 2020 En')
debug(final_r2015En.info(), 'info 2015 En')
debug(final_r2015Es.info(), 'info 2015 Es')


<class 'pandas.core.frame.DataFrame'>
Int64Index: 9990 entries, 0 to 998
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   precision          9990 non-null   float64
 1   recall             9990 non-null   float64
 2   f1-score           9990 non-null   float64
 3   accuracy           9990 non-null   float64
 4   auc                9990 non-null   float64
 5   features           9990 non-null   object 
 6   training time (s)  9990 non-null   float64
dtypes: float64(6), object(1)
memory usage: 624.4+ KB
-------------------------------------------------------------------
info 2020 En
None
-------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
Int64Index: 3996 entries, 0 to 998
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   precision          3996 non-null   float64
 1   r

Save the consolidated DataFrames

In [ ]:
final_r2020En.to_csv('2020English.csv', index=False)

In [ ]:
final_r2015En.to_csv('2015English.csv', index=False)

In [ ]:
final_r2015Es.to_csv('2015Spanish.csv', index=False)

In [ ]:
df_to_filter = final_r2015En

Example of the top auc experiments from CLEF-PAN 2015 in English (all from the baseline NAS)

In [ ]:
df_to_filter[:5]

,precision,recall,f1-score,accuracy,auc,features,training time (s)
0,0.600000,1.000000,0.75,0.733333,0.777778,"Chunks 500 B-A MLP with solver adam, ...",386.671067
0,0.875000,0.583333,0.70,0.800000,0.763889,"Chunks 100 A-B MLP with solver adam, ...",490.413687
0,0.875000,0.583333,0.70,0.800000,0.763889,"Chunks 200 A-B MLP with solver adam, ...",452.027749
1,0.692308,0.750000,0.72,0.766667,0.763889,"Chunks 100 A-B MLP with solver lbfgs, ...",490.413687
0,0.875000,0.583333,0.70,0.800000,0.763889,"Chunks 200 B-A MLP with solver sgd, ...",479.217417


We also get the results from the traditional methods to see what they obtained.

In [ ]:
nb_df = df_to_filter.loc[df_to_filter['features'].str.contains('naive Bayes')]

In [ ]:
svm_df = df_to_filter.loc[df_to_filter['features'].str.contains('SVC SVM')]

In [ ]:
log_reg_df = df_to_filter.loc[df_to_filter['features'].str.contains('Logistic Regression Classifier')]

In [ ]:
traditional_results = pd.concat([nb_df[:1], svm_df[:1], log_reg_df[:1]]).sort_values('auc', ascending=False)
traditional_results

,precision,recall,f1-score,accuracy,auc,features,training time (s)
39,0.500000,1.000000,0.666667,0.600000,0.666667,Chunks 400 A-B Default SVC SVM,0.019440
40,0.529412,0.750000,0.620690,0.633333,0.652778,Chunks 400 B-A Default Bernoulli naive Bayes,0.036537
249,0.423077,0.916667,0.578947,0.466667,0.541667,Chunks 100 B-A Logistic Regression Classifier ...,0.119041


This was the original logistic regression we used which allowed for more complexity, we followed the literature to build it.

In [ ]:

def log_reg(X_trainc, y_train, X_testc, y_test, sub_order):

    start_time = time.time()

    if sub_order == 2:
        X_trainc, X_testc = -X_trainc, -X_testc

    clf = SGDClassifier(loss='log', alpha=0.01)
    param_dist = {'alpha': loguniform(1e-4, 1e0)}
    n_iter_search = 2
    random_search = RandomizedSearchCV(clf, param_distributions=param_dist, n_iter=n_iter_search, verbose=2)
    random_search.fit(X_trainc, y_train)

    debug('', 'tuning alpha complete')

    clf = SGDClassifier(loss='log', alpha=random_search.best_params_['alpha'])
    num_epochs = 50
    aucs = []

    max_acc = -1
    best_model = None

    experiments_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy'])

    for i in trange(num_epochs):
        #print('Epoch - ', i)
        #print('-' * 30)

        clf.fit(X_trainc, y_train)

        """probs = clf.predict_proba(X)[:, 1]
        fpr, tpr, thresh = roc_curve(Y, probs)
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)"""

        preds = clf.predict(X_testc)

        report_dict = classification_report(preds, y_test, output_dict=True)

        curr_exp_df = pd.DataFrame(columns=['precision', 'recall', 'f1-score', 'accuracy'])
        curr_exp_df['precision'] = [report_dict['1']['precision']]
        curr_exp_df['recall'] = [report_dict['1']['recall']]
        curr_exp_df['f1-score'] = [report_dict['1']['f1-score']]
        curr_exp_df['accuracy'] = [report_dict['accuracy']]
        curr_exp_df['features'] = ['SGD with tuned alpha and log loss (logistic regression)']

        experiments_df = pd.concat([experiments_df, curr_exp_df])

        #acc = accuracy_score(y_test, preds)
        """if acc > max_acc:
            max_acc = acc
            best_model = clf
            best_preds = preds"""

    experiments_df['training time (s)'] = time.time() - start_time

    return experiments_df.sort_values('accuracy', ascending=False)#, time.time() - start_time


Analyzing the combinations

The size of the search space for baseline NAS is of 996 possible hyperparameter configurations.

In [ ]:
import itertools
n_neurons = [4, 8, 16, 32, 64, 128]
cnt = 0
for classifier_head in list(itertools.combinations_with_replacement(n_neurons, 1)) + \
            list(itertools.combinations_with_replacement(n_neurons, 2)) + \
            list(itertools.combinations_with_replacement(n_neurons, 3)):
    cnt += 1
print(cnt)

83


In [ ]:
83 * 12

996